# P03B — Audit dan eksplorasi dataset

Pendamping Bab 3 (subbab 3.2–3.9). Notebook ini memeriksa skema, ID, nilai hilang, duplikasi, rentang, statistik deskriptif, korelasi Pearson/Spearman, dan mutual information, lalu membuat **salinan rusak sementara** untuk latihan diagnosis.

**Cara menjalankan**
1. Jalankan `P03A_generator_data_sintesis_beton.ipynb` lebih dahulu. Di Colab, jalankan P03A dan P03B pada sesi yang sama; berkas ditulis ke `praktikum/bab03/`. Di komputer lokal, jalankan dari akar repositori.
2. Jalankan sel secara berurutan. Notebook ini hanya membaca CSV bersih dan tidak menimpanya.

Data bersifat sintesis. Angka di sini menjelaskan persamaan generator, bukan perilaku beton lapangan.

In [1]:
from pathlib import Path
import numpy as np, pandas as pd
from scipy.stats import pearsonr, spearmanr
from sklearn.feature_selection import mutual_info_regression

# Mencari CSV: akar repositori, folder unggahan Colab versi lama, atau folder kerja.
kandidat = [Path('praktikum/bab03'), Path('/content/Bab3_Paket/Data'), Path('.')]
folder = next((p for p in kandidat if (p / 'data_sintesis_beton.csv').exists()), None)
if folder is None:
    raise FileNotFoundError('data_sintesis_beton.csv tidak ditemukan. Jalankan P03A lebih dahulu, '
                            'atau jalankan notebook ini dari akar repositori.')
df = pd.read_csv(folder / 'data_sintesis_beton.csv')
print('Berkas dibaca dari:', folder / 'data_sintesis_beton.csv')

Berkas dibaca dari: praktikum/bab03/data_sintesis_beton.csv


## 1. Audit skema, mutu, dan statistik deskriptif (subbab 3.3 dan 3.9)

In [2]:
print('ukuran tabel      :', df.shape)
print('ID unik           :', df.id_spesimen.nunique())
print('baris duplikat    :', int(df.duplicated().sum()))
print('sel hilang        :', int(df.isna().sum().sum()))
print('\nrentang tiap kolom')
print(df.drop(columns='id_spesimen').agg(['min', 'max']).T.to_string())
print('\njumlah spesimen per umur (hari)')
print(df.umur_hari.value_counts().sort_index().to_string())
print('\nstatistik deskriptif')
print(df.drop(columns='id_spesimen').describe().T.round(3).to_string())
y = df.kuat_tekan_MPa
print(f'\ntarget: rata-rata {y.mean():.2f}, median {y.median():.2f}, simpangan baku (sampel) {y.std():.2f}, '
      f'rentang {y.min():.2f}-{y.max():.2f} MPa; nilai tepat 5,00 MPa: {int((y == 5).sum())} baris (batas bawah generator)')

ukuran tabel      : (1200, 10)
ID unik           : 1200
baris duplikat    : 0
sel hilang        : 0

rentang tiap kolom
                           min       max
semen_kg_m3             250.20   480.000
air_kg_m3                78.00   382.200
rasio_air_semen           0.30     0.819
fly_ash_kg_m3             0.00   120.000
superplasticizer_kg_m3    0.07    18.000
agregat_kasar_kg_m3     850.10  1149.800
agregat_halus_kg_m3     600.30   899.200
umur_hari                 3.00    90.000
kuat_tekan_MPa            5.00    53.100

jumlah spesimen per umur (hari)
umur_hari
3     150
7     228
14    178
28    328
56    133
90    183

statistik deskriptif
                         count     mean     std     min      25%      50%       75%       max
semen_kg_m3             1200.0  365.384  64.976  250.20  309.925  364.800   420.750   480.000
air_kg_m3               1200.0  204.436  65.486   78.00  155.275  196.050   248.925   382.200
rasio_air_semen         1200.0    0.560   0.148    0.30    0.43

## 2. Memeriksa fitur turunan: rasio air-semen (subbab 3.7)

Rasio dihitung dari air dibagi semen, tetapi CSV menyimpan ketiganya setelah dibulatkan. Selisih hasil hitung ulang bukan otomatis kesalahan. Toleransi dihitung dari pembulatan: rasio dibulatkan ke 0,001 (galat ≤ 0,0005), air dan semen ke 0,1 kg/m³ (galat ≤ 0,05).

In [3]:
hitung_ulang = df.air_kg_m3 / df.semen_kg_m3
selisih = (df.rasio_air_semen - hitung_ulang).abs()
toleransi_naif = 0.0005                                          # hanya pembulatan rasio
toleransi = 0.0005 + (0.05 / df.semen_kg_m3) * (1 + df.rasio_air_semen)   # + pembulatan air dan semen
print('selisih terbesar            :', round(selisih.max(), 5))
print('baris di atas 0,0005        :', int((selisih > toleransi_naif).sum()), 'dari', len(df), '(alarm palsu bila toleransi terlalu ketat)')
print('baris di atas toleransi total:', int((selisih > toleransi).sum()))

selisih terbesar            : 0.00071
baris di atas 0,0005        : 96 dari 1200 (alarm palsu bila toleransi terlalu ketat)
baris di atas toleransi total: 0


## 3. Keterkaitan fitur dengan target (subbab 3.6)

Korelasi dan mutual information (MI) hanya menggambarkan generator yang diketahui. MI diperkirakan dari data (di sini `mutual_info_regression` dengan `random_state=30303`), sehingga nilainya bergantung pada estimator dan ukuran data; baca urutannya, bukan angka mutlaknya.

In [4]:
fitur = ['semen_kg_m3', 'air_kg_m3', 'rasio_air_semen', 'fly_ash_kg_m3',
         'superplasticizer_kg_m3', 'agregat_kasar_kg_m3', 'agregat_halus_kg_m3', 'umur_hari']
baris = []
for x in fitur:
    r, _ = pearsonr(df[x], df.kuat_tekan_MPa)
    rs, _ = spearmanr(df[x], df.kuat_tekan_MPa)
    baris.append([x, r, rs])
korelasi = pd.DataFrame(baris, columns=['fitur', 'pearson_r', 'spearman_r'])
korelasi['mutual_info'] = mutual_info_regression(df[fitur], df.kuat_tekan_MPa, random_state=30303)
korelasi.iloc[:, :3].rename(columns={'fitur': 'feature'}).to_csv(folder / 'P03_correlations_synthetic.csv', index=False)
print('keterkaitan terhadap kuat_tekan_MPa (hanya deskripsi, bukan sebab-akibat)')
print(korelasi.round(3).to_string(index=False))

keterkaitan terhadap kuat_tekan_MPa (hanya deskripsi, bukan sebab-akibat)
                 fitur  pearson_r  spearman_r  mutual_info
           semen_kg_m3      0.565       0.557        0.226
             air_kg_m3     -0.232      -0.251        0.098
       rasio_air_semen     -0.655      -0.652        0.319
         fly_ash_kg_m3      0.216       0.204        0.059
superplasticizer_kg_m3      0.084       0.081        0.000
   agregat_kasar_kg_m3     -0.183      -0.178        0.032
   agregat_halus_kg_m3     -0.165      -0.159        0.000
             umur_hari      0.219       0.213        0.020


## 4. Latihan diagnosis pada salinan rusak (subbab 3.4 dan 3.9.1)

Tiga cacat dibuat **hanya pada salinan di memori**; CSV sumber tidak diubah: 20 nilai air dikosongkan, satu baris digandakan, dan satu nilai target dibuat ekstrem. Posisi cacat dipilih tetap (20 baris pertama, baris ke-51, baris ke-101) agar hasil dapat diulang. Karena itu pola ini **bukan** contoh MCAR, MAR, atau MNAR; jangan menyimpulkan mekanisme hilangnya data dari sini.

In [5]:
a = df.copy()
a.loc[0:19, 'air_kg_m3'] = np.nan                                   # 20 nilai air dikosongkan
a = pd.concat([a, a.iloc[[50]]], ignore_index=True)                 # baris S0051 digandakan
a.loc[100, 'kuat_tekan_MPa'] = a.kuat_tekan_MPa.max() + 50          # satu nilai target dibuat ekstrem

print('1) Nilai hilang per kolom:')
hilang = a.isna().sum()
print(hilang[hilang > 0].to_string(), '| indeks baris:', a.index[a.isna().any(axis=1)].min(), '-', a.index[a.isna().any(axis=1)].max())
print('\n2) Baris duplikat persis:')
print(a[a.duplicated(keep=False)][['id_spesimen', 'semen_kg_m3', 'kuat_tekan_MPa']].to_string())
print('   ID ganda:', a.id_spesimen[a.id_spesimen.duplicated()].tolist())

def pagar_iqr(s):
    q1, q3 = s.quantile([.25, .75]); i = q3 - q1
    return q1 - 1.5 * i, q3 + 1.5 * i

print('\n3) Penanda pencilan aturan IQR (Q1-1,5 IQR sampai Q3+1,5 IQR) pada kuat_tekan_MPa:')
for nama, data in [('salinan rusak', a), ('data bersih ', df)]:
    lo, hi = pagar_iqr(data.kuat_tekan_MPa)
    t = data[(data.kuat_tekan_MPa < lo) | (data.kuat_tekan_MPa > hi)]
    print(f'   {nama}: pagar {lo:.2f}-{hi:.2f}; ditandai {len(t)} baris ->', t.kuat_tekan_MPa.round(2).tolist())

1) Nilai hilang per kolom:
air_kg_m3    20 | indeks baris: 0 - 19

2) Baris duplikat persis:
     id_spesimen  semen_kg_m3  kuat_tekan_MPa
50         S0051        375.4            33.1
1200       S0051        375.4            33.1
   ID ganda: ['S0051']

3) Penanda pencilan aturan IQR (Q1-1,5 IQR sampai Q3+1,5 IQR) pada kuat_tekan_MPa:
   salinan rusak: pagar 1.47-51.55; ditandai 3 baris -> [103.1, 53.1, 51.72]
   data bersih : pagar 1.49-51.51; ditandai 2 baris -> [53.1, 51.72]


**Cara membaca keluaran** — jangan langsung memperbaiki:
- *Nilai hilang*: kolom mana, berapa baris, dan apa penyebabnya? Pada data ini air dapat dihitung dari `semen_kg_m3 × rasio_air_semen`, tetapi hubungan seperti itu harus dipastikan dulu pada data nyata.
- *Duplikasi*: apakah ID sama dan seluruh kolom sama (salinan) atau hanya nilai mirip (pengujian ulang)?
- *Pencilan*: aturan IQR hanya penanda. Pada data bersih pun dua nilai tertinggi ditandai padahal sah menurut generator, sedangkan nilai 5,00 MPa akibat batas bawah generator tidak ditandai. Tentukan informasi apa yang perlu diperiksa sebelum memperbaiki, mempertahankan, menandai, atau mengeluarkan rekaman.

Setelah memutuskan, catat jumlah baris sebelum dan sesudah, serta alasan setiap keputusan (subbab 3.4).

In [6]:
# Ilustrasi: memulihkan air yang hilang dari semen x rasio, lalu membandingkan dengan nilai aslinya.
# Hanya mungkin karena DGP diketahui; catat metode dan asumsinya bila dipakai pada data nyata.
pulih = (a.semen_kg_m3 * a.rasio_air_semen).loc[0:19]
galat = (pulih - df.air_kg_m3.loc[0:19]).abs()
print('galat pemulihan air (kg/m3): rata-rata', round(galat.mean(), 3), ', maksimum', round(galat.max(), 3))
print('Salinan ini tidak disimpan; CSV sumber tidak diubah.')

galat pemulihan air (kg/m3): rata-rata 0.085 , maksimum 0.219
Salinan ini tidak disimpan; CSV sumber tidak diubah.
